# Regime Analysis: End-to-End Pipeline

**Load data → compute Wasserstein distance matrix → detect regimes → visualise barycenters → displacement interpolation between regimes.**

This notebook demonstrates the complete workflow for Wasserstein-based regime detection.


In [1]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from sklearn.metrics import adjusted_rand_score

from data.synthetic import generate_regime_switching_panel
from otreturns.distributions import DistributionPanel
from otreturns.regimes import WassersteinRegimeDetector
from otreturns.barycenters import regime_barycenters
from otreturns.interpolation import interpolation_path
from otreturns.distances import wasserstein_1d
print("Imports OK")


Imports OK


## 1. Generate synthetic regime-switching panel

In [2]:
panel_df, true_labels = generate_regime_switching_panel(
    n_dates=90, n_stocks=300, seed=7
)
panel = DistributionPanel.from_panel(panel_df, min_stocks=50, winsorize=0.005)
dates = panel.dates
T = len(dates)
true_labels = true_labels[:T]
print(f"Panel: {T} dates")
print(f"Regime sizes: {np.bincount(true_labels).tolist()}")
print(f"Regime 0: normal bull  |  Regime 1: Student-t crisis  |  Regime 2: normal bear")


Panel: 90 dates
Regime sizes: [30, 30, 30]
Regime 0: normal bull  |  Regime 1: Student-t crisis  |  Regime 2: normal bear


## 2. Pairwise Wasserstein Distance Matrix

We compute $D_{st} = W_2(\mu_s, \mu_t)$ for all $s,t$.  Dates within the same regime should be close; dates in different regimes should be far apart.  The block structure in the heatmap reveals regime clustering.


In [3]:
detector = WassersteinRegimeDetector(n_regimes=3)
detector.fit(panel)
D = detector.distance_matrix

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(D, aspect="auto", cmap="viridis_r")
plt.colorbar(im, ax=ax, label="W₂")

# Mark regime boundaries
boundaries = np.where(np.diff(true_labels))[0] + 1
for b in boundaries:
    ax.axvline(b - 0.5, color="red", lw=1.5, alpha=0.7)
    ax.axhline(b - 0.5, color="red", lw=1.5, alpha=0.7)
ax.set_title("Wasserstein distance matrix (red = regime boundaries)")
ax.set_xlabel("Date"); ax.set_ylabel("Date")
plt.tight_layout()
plt.savefig("../figures/nb03_distance_matrix.png", dpi=100)
plt.show()
print(f"D shape: {D.shape}  |  max: {D.max():.4f}  |  mean off-diag: {D[~np.eye(T,dtype=bool)].mean():.4f}")


D shape: (90, 90)  |  max: 0.0187  |  mean off-diag: 0.0079


## 3. Spectral Clustering → Regime Labels

Spectral clustering on the Gaussian affinity matrix $A_{st} = \exp(-D_{st}^2 / 2\sigma^2)$ (median-heuristic bandwidth) assigns each date to a regime.


In [4]:
pred_labels = detector._labels  # already fitted above
ari = adjusted_rand_score(true_labels, pred_labels)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, labels, title in zip(axes,
    [true_labels, pred_labels],
    ["True regime labels", f"Detected labels  (ARI = {ari:.3f})"]):
    ax.scatter(range(T), labels, c=labels, cmap="tab10", vmin=0, vmax=9, s=12, alpha=0.9)
    ax.set_ylabel("Regime"); ax.set_title(title)
axes[-1].set_xlabel("Date index")
plt.tight_layout()
plt.savefig("../figures/nb03_regime_labels.png", dpi=100)
plt.show()
print(f"ARI = {ari:.4f}")


ARI = 0.9665


## 4. Regime Barycenters

The **Wasserstein barycenter** of a set of distributions is the distribution minimising the total squared W₂ distance.  In 1-d it has a beautiful closed form: the barycenter's quantile function is the weighted average of the input quantile functions (Agueh & Carlier 2011).

Each regime barycenter is the "canonical" distribution for that regime.


In [5]:
bary = regime_barycenters(panel, pred_labels, method="1d", n_support=300)
u = np.linspace(0.01, 0.99, 300)
palette = ["steelblue", "darkorange", "forestgreen"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Quantile functions
for lbl, b in sorted(bary.items()):
    m = b.moments(2)
    axes[0].plot(b.quantile_function(u), u, color=palette[lbl], lw=2.5,
                 label=f"Regime {lbl}  μ={m['mean']:.4f}  σ={np.sqrt(m['variance']):.4f}")
axes[0].set_xlabel("Return"); axes[0].set_ylabel("Quantile u")
axes[0].set_title("Barycenter quantile functions")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# PDFs (KDE)
def kde(s, x, bw=None):
    bw = bw or 1.06*np.std(s)*len(s)**(-0.2)
    d = (x[:,None]-s[None,:])/bw
    return np.mean(np.exp(-0.5*d**2),axis=1)/(bw*np.sqrt(2*np.pi))

x_grid = np.linspace(-0.08, 0.08, 300)
for lbl, b in sorted(bary.items()):
    dens = kde(b.samples, x_grid)
    axes[1].fill_between(x_grid, dens, alpha=0.2, color=palette[lbl])
    axes[1].plot(x_grid, dens, color=palette[lbl], lw=2.5, label=f"Regime {lbl}")
axes[1].set_xlabel("Return"); axes[1].set_ylabel("Density")
axes[1].set_title("Barycenter PDFs")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/nb03_barycenters.png", dpi=100)
plt.show()


## 5. Displacement Interpolation Between Regimes

The W₂-geodesic between two regime barycenters shows how the distribution "morphs" from one market state to another.  Each step is a valid probability distribution, and the path has constant speed.


In [6]:
lbls = sorted(bary.keys())
pairs = [(i,j,wasserstein_1d(bary[i],bary[j])) for i in lbls for j in lbls if j>i]
src_l, tgt_l, w2_total = max(pairs, key=lambda x: x[2])

path = interpolation_path(bary[src_l], bary[tgt_l], n_steps=8, n_support=300)
t_vals = np.linspace(0, 1, len(path))
cmap = cm.RdYlGn_r

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for k, (t, dist) in enumerate(zip(t_vals, path)):
    axes[0].plot(dist.quantile_function(u), u, color=cmap(t), lw=2, alpha=0.85)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0,1))
plt.colorbar(sm, ax=axes[0], label="t")
axes[0].set_xlabel("Return"); axes[0].set_ylabel("Quantile u")
axes[0].set_title(f"W₂-geodesic: regime {src_l} → regime {tgt_l}")
axes[0].grid(alpha=0.3)

# Mean and std along path
means = [d.moments(2)["mean"] for d in path]
stds  = [np.sqrt(d.moments(2)["variance"]) for d in path]
axes[1].plot(t_vals, means, "o-", color="steelblue",   lw=2, label="Mean (linear)")
axes[1].plot(t_vals, stds,  "s-", color="darkorange",  lw=2, label="Std (linear in W₂)")
axes[1].set_xlabel("t"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title("Moments along geodesic")

plt.tight_layout()
plt.savefig("../figures/nb03_interpolation.png", dpi=100)
plt.show()
print(f"W₂(regime {src_l}, regime {tgt_l}) = {w2_total:.5f}")
print("W₂ averages std devs — not variances (Agueh-Carlier barycenter property)")


W₂(regime 0, regime 1) = 0.01502
W₂ averages std devs — not variances (Agueh-Carlier barycenter property)


## 6. Transition Matrix & Silhouette Analysis

The **regime transition matrix** gives the empirical probability of moving from regime $i$ to regime $j$:
$P_{ij} = P(k_{t+1}=j | k_t=i)$

**Silhouette scores** measure cluster quality: close to 1 = well-separated regimes, close to 0 = ambiguous.


In [7]:
P = detector.transition_matrix()
sil = detector.silhouette_analysis()

print("Transition matrix:")
print("  " + "  ".join(f"R{j}" for j in range(3)))
for i, row in enumerate(P):
    print(f"R{i}" + "".join(f"  {v:.3f}" for v in row))

print(f"\nSilhouette scores:")
print(f"  Overall mean: {sil['mean_score']:.4f}")
for lbl, sc in sorted(sil['per_regime_scores'].items()):
    print(f"  Regime {lbl}:  {sc:.4f}")
print("Done.")


Transition matrix:
  R0  R1  R2
R0  0.967  0.033  0.000
R1  0.000  0.931  0.069
R2  0.000  0.033  0.967

Silhouette scores:
  Overall mean: 0.6270
  Regime 0:  0.7477
  Regime 1:  0.5905
  Regime 2:  0.5443
Done.
